# Pairwise Ranking Models for HDB Flat Recommendation

Each row in our data is a pair of flats (A, B) for a given user.  
Features: 9 user preference weights + 27 difference features (`diff_*` = score_A - score_B).  
Label: binary (1 = flat A preferred, 0 = flat B preferred), generated via Bradley-Terry.  

**Models:**
1. Logistic Regression (oracle baseline)
2. Gradient Boosting (HistGBM)
3. SVM (Linear + RBF kernel)
4. Gaussian Process Classifier
5. Neural Network / RankNet (MLP)
6. Random Forest

## Setup & Data Loading

In [15]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss, brier_score_loss
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("df_pairwise.csv")
print(f"Shape: {df.shape}")
print(f"Label balance:\n{df['label'].value_counts()}")
df.head()

Shape: (60000, 32)
Label balance:
label
0    30073
1    29927
Name: count, dtype: int64


,Unnamed: 0,user_id,flat_id_a,flat_id_b,label,weight_transport,weight_essentials,weight_lifestyle,weight_recreation,weight_health,...,diff_cafe_count_500m,diff_park_distance_m,diff_gym_distance_m,diff_doctor_distance_m,diff_pharmacy_distance_m,diff_storey_mid,diff_remaining_lease_years,diff_floor_area_sqm,diff_office_distance_m,diff_parents_distance_m
0,0,U1,669_HOUGANG AVE 8_4 ROOM_,103_HOUGANG AVE 1_4 ROOM_,1,5,4,2,3,1,...,2,472.719151,-1137.149655,318.883704,366.254051,3.0,3.833333,-5.0,1594.246497,-941.258949
1,1,U1,424C_YISHUN AVE 11_4 ROOM_,429B_YISHUN AVE 11_4 ROOM_,0,5,4,2,3,1,...,0,-60.364779,-123.985874,-112.612030,-98.610306,0.0,0.000000,0.0,171.680037,141.175071
2,2,U1,418_CANBERRA RD_4 ROOM_,508A_WELLINGTON CIRCLE_4 ROOM_,1,5,4,2,3,1,...,1,73.292998,74.118660,-24.244210,-13.702818,0.0,-3.416667,0.0,7.170500,77.027042
3,3,U1,136_BT BATOK WEST AVE 6_4 ROOM_,291A_BT BATOK ST 24_4 ROOM_,1,5,4,2,3,1,...,0,139.896394,-121.602443,47.468297,-60.783071,-9.0,-13.166667,-7.0,1480.694784,1095.842484
4,4,U1,621_CHOA CHU KANG ST 62_4 ROOM_,476D_CHOA CHU KANG AVE 5_4 ROOM_,0,5,4,2,3,1,...,2,-982.880807,1163.136025,610.726885,98.208540,-3.0,-17.166667,15.0,1057.239362,-674.977657


## Feature & Label Setup

In [16]:
weight_cols = [
    "weight_transport", "weight_essentials", "weight_lifestyle",
    "weight_recreation", "weight_health", "weight_storey",
    "weight_lease", "weight_office", "weight_parents",
]

diff_cols = [c for c in df.columns if c.startswith("diff_")]

feature_cols = weight_cols + diff_cols
print(f"Features ({len(feature_cols)}): {feature_cols}")

Features (27): ['weight_transport', 'weight_essentials', 'weight_lifestyle', 'weight_recreation', 'weight_health', 'weight_storey', 'weight_lease', 'weight_office', 'weight_parents', 'diff_mrt_distance_m', 'diff_bus_distance_m', 'diff_bus_count_500m', 'diff_grocery_or_supermarket_distance_m', 'diff_shopping_mall_distance_m', 'diff_restaurant_distance_m', 'diff_restaurant_count_500m', 'diff_cafe_distance_m', 'diff_cafe_count_500m', 'diff_park_distance_m', 'diff_gym_distance_m', 'diff_doctor_distance_m', 'diff_pharmacy_distance_m', 'diff_storey_mid', 'diff_remaining_lease_years', 'diff_floor_area_sqm', 'diff_office_distance_m', 'diff_parents_distance_m']


## Train / Validation Split (by user)

In [17]:
user_ids = df["user_id"].unique()
train_users, val_users = train_test_split(user_ids, test_size=0.2, random_state=42)

train_df = df[df["user_id"].isin(train_users)].copy()
val_df = df[df["user_id"].isin(val_users)].copy()

X_train = train_df[feature_cols].values
y_train = train_df["label"].values
X_val = val_df[feature_cols].values
y_val = val_df["label"].values

print(f"Train: {X_train.shape}, Val: {X_val.shape}")
print(f"Train label balance: {np.bincount(y_train)}")
print(f"Val label balance:   {np.bincount(y_val)}")

Train: (48000, 27), Val: (12000, 27)
Train label balance: [24052 23948]
Val label balance:   [6021 5979]


In [18]:
# Scaled versions for SVM, GP, MLP
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

## Evaluation Helpers

In [19]:
results = {}

def evaluate_model(name, y_true, y_pred_proba, train_time):
    """Compute pairwise classification metrics."""
    y_pred = (y_pred_proba >= 0.5).astype(int)
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred_proba)
    ll = log_loss(y_true, y_pred_proba)
    brier = brier_score_loss(y_true, y_pred_proba)

    results[name] = {
        "Accuracy": acc,
        "AUC-ROC": auc,
        "Log-Loss": ll,
        "Brier Score": brier,
        "Train Time (s)": train_time,
    }
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"  Accuracy:    {acc:.4f}")
    print(f"  AUC-ROC:     {auc:.4f}")
    print(f"  Log-Loss:    {ll:.4f}")
    print(f"  Brier Score: {brier:.4f}")
    print(f"  Train Time:  {train_time:.2f}s")

---
## Model 1: Logistic Regression (Oracle Baseline)

The Bradley-Terry generative process is sigmoid(linear combination of diffs × weights), which is exactly logistic regression. This model should nearly perfectly recover the data-generating process.

**Key experiment:** with vs. without interaction features (weight_i × diff_j).

In [20]:
from sklearn.linear_model import LogisticRegression

# --- 1a. Without interaction features ---
t0 = time.time()
lr_basic = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs')
lr_basic.fit(X_train, y_train)
t_basic = time.time() - t0

proba_basic = lr_basic.predict_proba(X_val)[:, 1]
evaluate_model("Logistic Regression (no interactions)", y_val, proba_basic, t_basic)


Logistic Regression (no interactions)
  Accuracy:    0.7748
  AUC-ROC:     0.8599
  Log-Loss:    0.4632
  Brier Score: 0.1528
  Train Time:  2.65s


In [21]:
# --- 1b. With interaction features (weight_i * diff_j) ---
# The true generative model is: sum(weight_i * score_i), so interactions are key

def add_interaction_features(X_df, weight_cols, diff_cols):
    """Create weight_i * diff_score_j interaction features."""
    X_int = X_df.copy()
    # Only interact weights with their corresponding score diffs
    score_diff_cols = [c for c in diff_cols if c.endswith('_score')]
    for wc in weight_cols:
        for dc in score_diff_cols:
            X_int[f"{wc}_x_{dc}"] = X_int[wc] * X_int[dc]
    return X_int

train_int = add_interaction_features(train_df[feature_cols], weight_cols, diff_cols)
val_int = add_interaction_features(val_df[feature_cols], weight_cols, diff_cols)

interaction_feature_cols = train_int.columns.tolist()
print(f"Features with interactions: {len(interaction_feature_cols)}")

t0 = time.time()
lr_inter = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs')
lr_inter.fit(train_int.values, y_train)
t_inter = time.time() - t0

proba_inter = lr_inter.predict_proba(val_int.values)[:, 1]
evaluate_model("Logistic Regression (with interactions)", y_val, proba_inter, t_inter)

Features with interactions: 27

Logistic Regression (with interactions)
  Accuracy:    0.7748
  AUC-ROC:     0.8599
  Log-Loss:    0.4632
  Brier Score: 0.1528
  Train Time:  2.77s


In [22]:
# Inspect learned coefficients for the interaction model
coef_df = pd.DataFrame({
    "feature": interaction_feature_cols,
    "coefficient": lr_inter.coef_[0]
}).sort_values("coefficient", ascending=False, key=abs)

print("Top 20 coefficients (by magnitude):")
coef_df.head(20)

Top 20 coefficients (by magnitude):


,feature,coefficient
15,diff_restaurant_count_500m,0.120995
17,diff_cafe_count_500m,0.099800
22,diff_storey_mid,0.060061
23,diff_remaining_lease_years,0.037374
11,diff_bus_count_500m,0.009602
3,weight_recreation,-0.006884
6,weight_lease,-0.006213
8,weight_parents,-0.005053
4,weight_health,-0.004706
7,weight_office,0.003685


---
## Model 2: Gradient Boosting (HistGBM)

sklearn's HistGradientBoostingClassifier (similar to LightGBM/XGBoost).  
Trees can automatically discover the interaction structure (weight × diff) without manual feature engineering.

In [23]:
from sklearn.ensemble import HistGradientBoostingClassifier

t0 = time.time()
hgb_clf = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=6,
    max_iter=200,
    random_state=42,
)
hgb_clf.fit(X_train, y_train)
t_hgb = time.time() - t0

proba_hgb = hgb_clf.predict_proba(X_val)[:, 1]
evaluate_model("Gradient Boosting (HistGBM)", y_val, proba_hgb, t_hgb)


Gradient Boosting (HistGBM)
  Accuracy:    0.7826
  AUC-ROC:     0.8749
  Log-Loss:    0.4422
  Brier Score: 0.1449
  Train Time:  1.98s


In [24]:
from sklearn.inspection import permutation_importance

perm_imp = permutation_importance(hgb_clf, X_val, y_val, n_repeats=10, random_state=42, n_jobs=-1)

importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm_imp.importances_mean,
    "importance_std": perm_imp.importances_std,
}).sort_values("importance_mean", ascending=False)

print("Top 15 features (permutation importance):")
importance_df.head(15)

Top 15 features (permutation importance):


,feature,importance_mean,importance_std
26,diff_parents_distance_m,0.036675,0.002901
25,diff_office_distance_m,0.027425,0.001970
12,diff_grocery_or_supermarket_distance_m,0.026883,0.003054
20,diff_doctor_distance_m,0.026667,0.001998
23,diff_remaining_lease_years,0.025658,0.001381
9,diff_mrt_distance_m,0.023617,0.001254
18,diff_park_distance_m,0.018158,0.001576
19,diff_gym_distance_m,0.015475,0.002132
22,diff_storey_mid,0.015350,0.002252
15,diff_restaurant_count_500m,0.014850,0.002104


---
## Model 3: SVM (Linear + RBF Kernel)

Linear kernel SVM on pairwise diffs = classic RankSVM (Joachims, 2002).  
RBF kernel tests for nonlinear preference boundaries.

**Note:** SVM training is slow (~2-7 min per kernel on this data).

In [30]:
from sklearn.svm import SVC

# --- 3a. Linear kernel ---
t0 = time.time()
svm_linear = SVC(kernel='linear', C=1.0, probability=True, random_state=42)
svm_linear.fit(X_train_scaled, y_train)
t_svm_lin = time.time() - t0

proba_svm_lin = svm_linear.predict_proba(X_val_scaled)[:, 1]
evaluate_model("SVM (Linear)", y_val, proba_svm_lin, t_svm_lin)


SVM (Linear)
  Accuracy:    0.7718
  AUC-ROC:     0.8596
  Log-Loss:    0.4637
  Brier Score: 0.1530
  Train Time:  726.88s


In [28]:
# --- 3b. RBF kernel ---
t0 = time.time()
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
svm_rbf.fit(X_train_scaled, y_train)
t_svm_rbf = time.time() - t0

proba_svm_rbf = svm_rbf.predict_proba(X_val_scaled)[:, 1]
evaluate_model("SVM (RBF)", y_val, proba_svm_rbf, t_svm_rbf)


SVM (RBF)
  Accuracy:    0.7903
  AUC-ROC:     0.8783
  Log-Loss:    0.4358
  Brier Score: 0.1422
  Train Time:  278.31s


---
## Model 4: Gaussian Process Classifier

Bayesian nonparametric with calibrated uncertainty estimates.  
Subsampled to 3000 training points due to O(n³) scaling.

**Note:** GP training takes ~1 min.

In [29]:
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

# Subsample for GP (O(n^3) scaling)
GP_TRAIN_SIZE = 3000
np.random.seed(42)
gp_idx = np.random.choice(len(X_train_scaled), size=min(GP_TRAIN_SIZE, len(X_train_scaled)), replace=False)
X_train_gp = X_train_scaled[gp_idx]
y_train_gp = y_train[gp_idx]
print(f"GP training on {len(X_train_gp)} samples (subsampled from {len(X_train_scaled)})")

t0 = time.time()
gp_clf = GaussianProcessClassifier(
    kernel=1.0 * RBF(length_scale=1.0) + WhiteKernel(noise_level=0.1),
    random_state=42,
    max_iter_predict=100,
)
gp_clf.fit(X_train_gp, y_train_gp)
t_gp = time.time() - t0

proba_gp = gp_clf.predict_proba(X_val_scaled)[:, 1]
evaluate_model("Gaussian Process", y_val, proba_gp, t_gp)

GP training on 3000 samples (subsampled from 48000)

Gaussian Process
  Accuracy:    0.7792
  AUC-ROC:     0.8694
  Log-Loss:    0.4488
  Brier Score: 0.1477
  Train Time:  98.97s


---
## Model 5: Neural Network / RankNet (MLP)

Feedforward MLP trained with binary cross-entropy on pairwise data.  
**Key experiment:** vary depth to study capacity vs. generalization.

In [25]:
from sklearn.neural_network import MLPClassifier

architectures = {
    "MLP (1 layer: 64)": (64,),
    "MLP (2 layers: 64-32)": (64, 32),
    "MLP (3 layers: 128-64-32)": (128, 64, 32),
}

mlp_probas = {}
for name, hidden in architectures.items():
    t0 = time.time()
    mlp = MLPClassifier(
        hidden_layer_sizes=hidden,
        activation='relu',
        solver='adam',
        learning_rate_init=0.001,
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=42,
    )
    mlp.fit(X_train_scaled, y_train)
    t_mlp = time.time() - t0

    proba_mlp = mlp.predict_proba(X_val_scaled)[:, 1]
    mlp_probas[name] = proba_mlp
    evaluate_model(name, y_val, proba_mlp, t_mlp)


MLP (1 layer: 64)
  Accuracy:    0.7877
  AUC-ROC:     0.8807
  Log-Loss:    0.4299
  Brier Score: 0.1413
  Train Time:  1.84s

MLP (2 layers: 64-32)
  Accuracy:    0.7878
  AUC-ROC:     0.8791
  Log-Loss:    0.4319
  Brier Score: 0.1421
  Train Time:  2.05s

MLP (3 layers: 128-64-32)
  Accuracy:    0.7837
  AUC-ROC:     0.8786
  Log-Loss:    0.4361
  Brier Score: 0.1438
  Train Time:  3.51s


---
## Model 6: Random Forest

Bagging-based ensemble baseline. Compares bagging vs. boosting (HistGBM).

In [26]:
from sklearn.ensemble import RandomForestClassifier

t0 = time.time()
rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
)
rf_clf.fit(X_train, y_train)
t_rf = time.time() - t0

proba_rf = rf_clf.predict_proba(X_val)[:, 1]
evaluate_model("Random Forest", y_val, proba_rf, t_rf)


Random Forest
  Accuracy:    0.7520
  AUC-ROC:     0.8409
  Log-Loss:    0.5043
  Brier Score: 0.1669
  Train Time:  3.47s


---
## Results Comparison

In [31]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values("AUC-ROC", ascending=False)
print("\nAll Models — Pairwise Classification Metrics")
print("=" * 70)
results_df


All Models — Pairwise Classification Metrics


,Accuracy,AUC-ROC,Log-Loss,Brier Score,Train Time (s)
MLP (1 layer: 64),0.787667,0.880662,0.429945,0.141341,1.835772
MLP (2 layers: 64-32),0.787833,0.879050,0.431922,0.142144,2.048139
MLP (3 layers: 128-64-32),0.783667,0.878638,0.436097,0.143841,3.509294
SVM (RBF),0.790333,0.878284,0.435818,0.142162,278.307814
Gradient Boosting (HistGBM),0.782583,0.874865,0.442232,0.144931,1.984874
Gaussian Process,0.779167,0.869393,0.448757,0.147685,98.973523
Logistic Regression (no interactions),0.774833,0.859851,0.463159,0.152828,2.649903
Logistic Regression (with interactions),0.774833,0.859851,0.463159,0.152828,2.767668
SVM (Linear),0.771750,0.859563,0.463691,0.153010,726.878592
Random Forest,0.752000,0.840890,0.504340,0.166858,3.467811


---
## Ranking Evaluation: NDCG@K

Convert pairwise predictions to per-flat rankings via win-rate aggregation,  
then compute NDCG@5 and NDCG@10 per user.

In [32]:
def dcg_at_k(relevances, k):
    relevances = np.asarray(relevances)[:k]
    if relevances.size == 0:
        return 0.0
    return np.sum((2 ** relevances - 1) / np.log2(np.arange(2, relevances.size + 2)))

def ndcg_at_k(y_true, y_score, k=5):
    order = np.argsort(y_score)[::-1]
    y_true_sorted = np.take(y_true, order)
    dcg = dcg_at_k(y_true_sorted, k)
    ideal = dcg_at_k(np.sort(y_true)[::-1], k)
    if ideal == 0:
        return 0.0
    return dcg / ideal


def pairwise_to_ranking_ndcg(val_df, pred_proba, k_values=[5, 10]):
    """
    Convert pairwise predictions to flat rankings via win-rate,
    then compute NDCG@K.
    """
    eval_df = val_df.copy()
    eval_df["pred_proba"] = pred_proba
    
    ndcg_results = {f"NDCG@{k}": [] for k in k_values}
    
    for user_id, group in eval_df.groupby("user_id"):
        flat_scores = {}
        label_scores = {}
        for _, row in group.iterrows():
            fa, fb = row["flat_id_a"], row["flat_id_b"]
            p = row["pred_proba"]
            lab = row["label"]
            flat_scores.setdefault(fa, []).append(p)
            flat_scores.setdefault(fb, []).append(1 - p)
            label_scores.setdefault(fa, []).append(lab)
            label_scores.setdefault(fb, []).append(1 - lab)
        
        flat_ranking = {fid: np.mean(scores) for fid, scores in flat_scores.items()}
        if len(flat_ranking) < 2:
            continue
        
        flat_ids = list(flat_ranking.keys())
        pred_scores = np.array([flat_ranking[fid] for fid in flat_ids])
        true_scores = np.array([np.mean(label_scores[fid]) for fid in flat_ids])
        
        for k in k_values:
            score = ndcg_at_k(true_scores, pred_scores, k=k)
            ndcg_results[f"NDCG@{k}"].append(score)
    
    return {metric: np.mean(vals) for metric, vals in ndcg_results.items()}

In [33]:
# Compute NDCG for all models
model_predictions = {
    "Logistic Regression (no interactions)": proba_basic,
    "Logistic Regression (with interactions)": proba_inter,
    "Gradient Boosting (HistGBM)": proba_hgb,
    "SVM (Linear)": proba_svm_lin,
    "SVM (RBF)": proba_svm_rbf,
    "Gaussian Process": proba_gp,
    "Random Forest": proba_rf,
}
model_predictions.update(mlp_probas)

ndcg_table = {}
for name, proba in model_predictions.items():
    ndcg = pairwise_to_ranking_ndcg(val_df, proba)
    ndcg_table[name] = ndcg
    print(f"{name}: NDCG@5={ndcg['NDCG@5']:.4f}, NDCG@10={ndcg['NDCG@10']:.4f}")

ndcg_df = pd.DataFrame(ndcg_table).T
ndcg_df.sort_values("NDCG@5", ascending=False)

Logistic Regression (no interactions): NDCG@5=0.9901, NDCG@10=0.9848
Logistic Regression (with interactions): NDCG@5=0.9901, NDCG@10=0.9848
Gradient Boosting (HistGBM): NDCG@5=0.9915, NDCG@10=0.9859
SVM (Linear): NDCG@5=0.9934, NDCG@10=0.9858
SVM (RBF): NDCG@5=0.9925, NDCG@10=0.9837
Gaussian Process: NDCG@5=0.9986, NDCG@10=0.9875
Random Forest: NDCG@5=0.9963, NDCG@10=0.9800
MLP (1 layer: 64): NDCG@5=0.9976, NDCG@10=0.9930
MLP (2 layers: 64-32): NDCG@5=0.9991, NDCG@10=0.9944
MLP (3 layers: 128-64-32): NDCG@5=0.9976, NDCG@10=0.9935


,NDCG@5,NDCG@10
MLP (2 layers: 64-32),0.999098,0.994387
Gaussian Process,0.998621,0.987472
MLP (1 layer: 64),0.997566,0.993009
MLP (3 layers: 128-64-32),0.997566,0.993549
Random Forest,0.996332,0.980049
SVM (Linear),0.993440,0.985812
SVM (RBF),0.992517,0.983746
Gradient Boosting (HistGBM),0.991521,0.985918
Logistic Regression (no interactions),0.990103,0.984768
Logistic Regression (with interactions),0.990103,0.984768


---
## Final Comparison Table

In [34]:
final_df = results_df.join(ndcg_df, how='left').sort_values("AUC-ROC", ascending=False)
print("Final Model Comparison (Pairwise + Ranking)")
print("=" * 80)
final_df

Final Model Comparison (Pairwise + Ranking)


,Accuracy,AUC-ROC,Log-Loss,Brier Score,Train Time (s),NDCG@5,NDCG@10
MLP (1 layer: 64),0.787667,0.880662,0.429945,0.141341,1.835772,0.997566,0.993009
MLP (2 layers: 64-32),0.787833,0.879050,0.431922,0.142144,2.048139,0.999098,0.994387
MLP (3 layers: 128-64-32),0.783667,0.878638,0.436097,0.143841,3.509294,0.997566,0.993549
SVM (RBF),0.790333,0.878284,0.435818,0.142162,278.307814,0.992517,0.983746
Gradient Boosting (HistGBM),0.782583,0.874865,0.442232,0.144931,1.984874,0.991521,0.985918
Gaussian Process,0.779167,0.869393,0.448757,0.147685,98.973523,0.998621,0.987472
Logistic Regression (no interactions),0.774833,0.859851,0.463159,0.152828,2.649903,0.990103,0.984768
Logistic Regression (with interactions),0.774833,0.859851,0.463159,0.152828,2.767668,0.990103,0.984768
SVM (Linear),0.771750,0.859563,0.463691,0.153010,726.878592,0.993440,0.985812
Random Forest,0.752000,0.840890,0.504340,0.166858,3.467811,0.996332,0.980049


---
## Save Models for Explainability Analysis

In [ ]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(hgb_clf, "models/hgb_clf.joblib")
joblib.dump(scaler, "models/scaler.joblib")

# Retrain the best MLP to save the model object (mlp_probas only stores predictions)
mlp_best = MLPClassifier(
    hidden_layer_sizes=(64,),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42,
)
mlp_best.fit(X_train_scaled, y_train)
joblib.dump(mlp_best, "models/mlp_1layer.joblib")

print("Saved: models/hgb_clf.joblib, models/mlp_1layer.joblib, models/scaler.joblib")

Saved: models/hgb_clf.joblib, models/mlp_1layer.joblib, models/scaler.joblib
